In [8]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Create map
Map = geemap.Map(center=[8.5, -80], zoom=7)

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

# Hydrographic basins
basins = ee.Image(
    "projects/deforestation-495419/assets/hydrographic_basins"
).clip(panama)

# Convert to integer basin IDs
basins_int = basins.toInt()

# Detect basin boundaries

# Compare neighboring pixels
neighbors = basins_int.focal_mode(radius=1)

# Boundary pixels = where neighbor ID differs
basin_edges = basins_int.neq(neighbors).selfMask()

# Distance to nearest basin boundary

# Compute distance from NON-edge pixels to nearest edge
distance_pixels = basin_edges.fastDistanceTransform(
    neighborhood=512,
    units='pixels'
).sqrt()

# Convert to meters
pixel_size = basins.projection().nominalScale()

distance_to_basin_m = (
    distance_pixels
    .multiply(pixel_size)
    .rename("distance_to_basin_m")
    .clip(panama)
)

# Visualization

# Map.addLayer(panama, {}, "Panama")

Map.addLayer(
    basins,
    {},
    "Hydrographic Basins"
)

Map.addLayer(
    basin_edges,
    {"palette": ["red"]},
    "Basin Boundaries"
)

Map.addLayer(
    distance_to_basin_m,
    {
        "min": 0,
        "max": 50000,
        "palette": ["blue", "cyan", "yellow", "red"]
    },
    "Distance to Basin Boundary (m)"
)

Map.centerObject(panama, 7)

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…